In [ ]:
res = !wb resource resolve --id bucket_for_storage_across_apps
bucket = res[0]
print(f"Bucket: {bucket}")

In [ ]:
import hail as hl
hl.stop()

In [ ]:
import os
project_id = os.environ.get("GOOGLE_PROJECT")
hl.init(idempotent=True,
       spark_conf={
        "spark.hadoop.fs.gs.project.id": project_id,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
        "spark.hadoop.fs.gs.requester.pays.project.id": project_id,
        "spark.task.maxFailures": "8",
        "spark.stage.maxConsecutiveAttempts": "8",
        "spark.network.timeout": "800s",
        "spark.executor.heartbeatInterval": "60s",
        "spark.rpc.askTimeout": "600s",
        "spark.rpc.lookupTimeout": "600s",
        "spark.hadoop.fs.gs.http.max.retry": "20",
        "spark.hadoop.fs.gs.http.connect-timeout": "60000",
        "spark.hadoop.fs.gs.http.read-timeout": "60000",
       },
        log='/tmp/hail.log',
        quiet=True
       )
hl.default_reference('GRCh38')

In [ ]:
vds_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/vds/hail.vds/"
vds = hl.vds.read_vds(vds_path)
print("Check1")

In [ ]:
bag3_interval = [hl.parse_locus_interval('chr10:119551811-119716826', reference_genome='GRCh38')]
vds_bag3 = hl.vds.filter_intervals(vds, bag3_interval)
print("Check2")

In [ ]:
mt_to_export = vds_bag3.variant_data
mt_vcf_ready = mt_to_export.select_entries(GT = mt_to_export.LGT)
mt_vcf_ready = mt_vcf_ready.select_rows()
mt_vcf_ready = mt_vcf_ready.select_globals()
print("Check3")

vcf_path = f'{bucket}/bag3_full_region.vcf.bgz'
hl.export_vcf(mt_vcf_ready, vcf_path, tabix=False)
print("Export done.")